# Control flow and functions

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/python-03-control-flow-and-functions.ipynb)

Pipelines are made of small functions: each one takes data in, checks it and returns data out. This chapter covers
the control-flow statements you need, how to write functions that are easy to call correctly, and the scope rules
that make closures and default arguments behave the way they do.

## `if`, `for` and `while`

`for` walks any iterable. `range(start, stop, step)` produces numbers, and the stop value is never included.
`enumerate` gives the index with each item, and `zip` walks two sequences together.

In [ ]:
total = 0
for n in range(1, 11, 2):          # 1, 3, 5, 7, 9
    total += n
assert total == 25

names = ["ana", "ben", "cy"]
for index, name in enumerate(names, start=1):
    if index == 2:
        assert name == "ben"

pairs = list(zip(names, [30, 25, 41]))
assert pairs[2] == ("cy", 41)

retries = 0
while retries < 3:
    retries += 1
assert retries == 3

## `break`, `continue` and `for ... else`

`break` leaves the loop. `continue` skips to the next item. The `else` block of a loop runs only when the loop finished
without `break`. It is the clean way to write "search, and do something if not found".

In [ ]:
def find_first_negative(values):
    for index, value in enumerate(values):
        if value < 0:
            found = index
            break
    else:
        found = None          # no break: nothing negative
    return found

assert find_first_negative([3, -1, -5]) == 1
assert find_first_negative([1, 2]) is None

kept = []
for value in [1, None, 3]:
    if value is None:
        continue
    kept.append(value)
assert kept == [1, 3]

## Defining functions

A function does one job, takes its inputs as arguments and returns its result. Write a **docstring** as the first line of
the body. Functions that return nothing return `None`.

Keyword arguments make calls readable. Arguments after a `*` must be passed by keyword, which prevents mixing up two
values of the same type.

In [ ]:
def load_window(start, end, *, inclusive_end=False):
    """Return the window as a half-open (start, stop) tuple."""
    stop = end + 1 if inclusive_end else end
    return (start, stop)

assert load_window(1, 5) == (1, 5)
assert load_window(1, 5, inclusive_end=True) == (1, 6)

try:
    load_window(1, 5, True)       # positional use of a keyword-only argument
except TypeError:
    pass
else:
    raise AssertionError("expected TypeError")

## `*args` and `**kwargs`

`*args` collects extra positional arguments into a tuple. `**kwargs` collects extra keyword arguments into a dict.
Use them to forward arguments, not to hide what a function accepts.

In [ ]:
def connect(host, *args, timeout=5, **options):
    return host, args, timeout, options

host, args, timeout, options = connect("db1", 1, 2, timeout=10, retries=3)
assert host == "db1"
assert args == (1, 2)
assert timeout == 10
assert options == {"retries": 3}

# Unpacking works in calls too.
settings = {"timeout": 30}
assert connect("db2", **settings)[2] == 30

## The mutable default argument trap

Default values are created **once**, when the function is defined. A mutable default such as `[]` is shared by every
call. Use `None` and create the object inside the function.

In [ ]:
def add_bad(item, bucket=[]):          # the list is created once
    bucket.append(item)
    return bucket

def add_good(item, bucket=None):
    if bucket is None:
        bucket = []
    bucket.append(item)
    return bucket

add_bad("a")
assert add_bad("b") == ["a", "b"]      # leaked state from the first call

assert add_good("a") == ["a"]
assert add_good("b") == ["b"]          # each call starts fresh

## Scope and closures

Names assigned inside a function are local. Reading a name that is not assigned locally looks in the enclosing
scope. To rebind an outer name, declare it with `nonlocal`, or return the new value. Avoid `global` in pipeline code.

A **closure** is an inner function that remembers variables from the enclosing function. It is a simple way to make a
configured function.

In [ ]:
def make_threshold_check(limit):
    def is_over(value):
        return value > limit      # `limit` is remembered by the closure
    return is_over

over_100 = make_threshold_check(100)
assert over_100(150) and not over_100(50)

def make_counter():
    count = 0
    def increment():
        nonlocal count
        count += 1
        return count
    return increment

counter = make_counter()
counter()
assert counter() == 2

## Lambdas and higher-order functions

A `lambda` is a small anonymous function for one expression. Use it as a `key=` argument. For anything longer than one
expression, write a named function. `map`, `filter` and `functools.reduce` take a function as an argument.

In [ ]:
from functools import reduce

rows = [{"amount": 3}, {"amount": 10}, {"amount": 7}]

assert list(map(lambda r: r["amount"] * 2, rows)) == [6, 20, 14]
assert [r["amount"] for r in rows if r["amount"] > 5] == [10, 7]   # a comprehension reads better
assert reduce(lambda acc, r: acc + r["amount"], rows, 0) == 20

## Practice questions

1. Write `first_duplicate(values)` that returns the first value seen twice, or `None`.
2. Write `make_batcher(items, size)` that returns a function. Each call returns the next `size` items of a list it holds, or `[]` when
   it is empty.
3. Fix this function so it does not share state between calls:
   ```python
   def record(event, seen=[]):
       seen.append(event)
       return seen
   ```

## Answers

### Question 1

In [ ]:
def first_duplicate(values):
    seen = set()
    for value in values:
        if value in seen:
            return value
        seen.add(value)
    return None

assert first_duplicate([2, 1, 3, 1, 2]) == 1
assert first_duplicate([1, 2]) is None

### Question 2

In [ ]:
def make_batcher(items, size):
    position = 0
    def next_batch():
        nonlocal position
        batch = items[position:position + size]
        position += size
        return batch
    return next_batch

next_batch = make_batcher(list(range(5)), 2)
assert next_batch() == [0, 1]
assert next_batch() == [2, 3]
assert next_batch() == [4]
assert next_batch() == []

### Question 3

In [ ]:
def record(event, seen=None):
    if seen is None:
        seen = []
    seen.append(event)
    return seen

assert record("a") == ["a"]
assert record("b") == ["b"]

## Interview notes

- **Use `for ... else` for search loops.** It makes the "not found" branch explicit.
- **Keyword-only arguments** (after `*`) make call sites self-documenting, which matters in pipelines with many options.
- **Mutable defaults are shared.** Always use `None` and create the object inside the function.
- **Closures capture variables, not values**, so a loop that builds closures needs a default argument to freeze each value.